# scattering

> Wavelet scattering along the time axis, in plain torch.

`01_data` arranges the exchange as `[channels, days]` (and, with several
orderings, `[views, channels, days]`). This notebook adds a third axis: for every
channel, a **wavelet scattering transform** along time turns one series into a
small stack of multi-scale, locally-stable descriptors, so the array becomes
`[channels, paths, days]` -- or `[views, channels, paths, days]` once the
orderings are stacked as views.

Why scattering rather than a raw wavelet transform: the modulus + low-pass at
each order makes the coefficients stable to small time-warps and averaging-window
shifts, while the second order recovers the cross-scale interactions that the
first-order averaging throws away (envelope-of-envelope: volatility clustering,
in finance terms). See Mallat (2012), Bruna & Mallat (2013), and for the
finance angle Morel et al. (2023), *Scale Dependencies and Self-Similar Models
with Wavelet Scattering Spectra*.

**Self-contained by design.** No `kymatio`, no `scipy`: a Morlet filter bank is
~30 lines and the transform is FFT-multiply-IFFT. The filter-bank construction
follows `kymatio` / Morel's `scatspectra` (both MIT/BSD -- reimplemented here,
not vendored). Every function stays device-agnostic and inherits its input's
device, like everything in `01_data`.

**Style note:** same as `01_data` -- one function per cell, then a test cell
with at least one `assert`, on small synthetic data.

In [ ]:
#| default_exp scattering

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
import matplotlib
matplotlib.use("Agg")
%matplotlib inline

In [ ]:
#| export
import math

import numpy as np
import torch

## Filter bank

A Morlet wavelet in Fourier is a Gaussian bump at frequency $\xi$ minus a
small multiple of a Gaussian at $0$, so that $\hat\psi(0) = 0$ exactly (zero
mean in time):

$$\hat\psi(\omega) = e^{-(\omega-\xi)^2/2\sigma^2} - \kappa\, e^{-\omega^2/2\sigma^2},
\qquad \kappa = \hat g(-\xi)/\hat g(0).$$

Frequencies are in cycles/sample, so the Nyquist limit is $0.5$. With `Q`
wavelets per octave and `J` octaves, the centre frequencies are
$\xi_j = \xi_0\, 2^{-j/Q}$ for $j = 0,\dots,JQ-1$, and $\sigma_j$ is chosen so
that adjacent bumps cross at $1/\sqrt{2}$ of their peak (the standard
`kymatio` rule). The low-pass $\hat\phi$ is a Gaussian with the bandwidth of
the *lowest*-frequency wavelet.

Two details that matter numerically:

- **Periodization.** A wide Gaussian in Fourier does not vanish at the edge
  of `[0, 1)`, so it is evaluated over `2P-1` periods and folded, otherwise
  the filter has a discontinuity at the Nyquist wrap-around.
- **L1 normalization in time**, so that the modulus of a wavelet coefficient
  is on the same scale as the signal regardless of $\sigma$.

In [ ]:
#| export
def _gauss_hat(freqs, sigma):
    return torch.exp(-freqs ** 2 / (2 * sigma ** 2))


def _periodize(h, nperiods):
    """Fold a Fourier-domain filter evaluated over `nperiods` periods back
    onto one period: `v[k] = mean_i h[i*N + k]`."""
    N = h.shape[-1] // nperiods
    return h.reshape(*h.shape[:-1], nperiods, N).mean(dim=-2)


def _l1_normalize(h_hat):
    """Scale so the filter's time-domain L1 norm is 1."""
    h = torch.fft.ifft(h_hat)
    return h_hat / h.abs().sum(dim=-1, keepdim=True)


def morlet_hat(N, xi, sigma, eps=1e-7):
    """Fourier transform of one L1-normalized Morlet wavelet on `N` samples,
    at the frequencies `torch.fft.fftfreq(N)`. `xi`, `sigma` in cycles/sample."""
    P = int(math.ceil(math.sqrt(-2 * sigma ** 2 * math.log(eps)) + 1))
    freqs = torch.arange((1 - P) * N, P * N, dtype=torch.float64) / N
    gabor = _periodize(_gauss_hat(freqs - xi, sigma), 2 * P - 1)
    low = _periodize(_gauss_hat(freqs, sigma), 2 * P - 1)
    kappa = gabor[0] / low[0]
    return _l1_normalize(gabor - kappa * low)


def gauss_hat(N, sigma, eps=1e-7):
    """Fourier transform of an L1-normalized Gaussian low-pass on `N` samples."""
    P = int(math.ceil(math.sqrt(-2 * sigma ** 2 * math.log(eps)) + 1))
    freqs = torch.arange((1 - P) * N, P * N, dtype=torch.float64) / N
    return _l1_normalize(_periodize(_gauss_hat(freqs, sigma), 2 * P - 1))

In [ ]:
N = 256
psi = morlet_hat(N, xi=0.25, sigma=0.05)
assert psi.shape == (N,)
assert abs(psi[0]) < 1e-12, "Morlet must have exactly zero mean"
assert abs(torch.fft.ifft(psi).abs().sum() - 1) < 1e-6, "L1 normalized in time"
# peaks at (or next to) xi
freqs = torch.fft.fftfreq(N)
assert abs(freqs[psi.abs().argmax()] - 0.25) <= 1 / N

phi = gauss_hat(N, sigma=0.02)
assert phi.abs().argmax() == 0 and abs(torch.fft.ifft(phi).abs().sum() - 1) < 1e-6

In [ ]:
#| export
def morlet_params(J, Q=1, high_freq=0.425, r=math.sqrt(0.5)):
    """Centre frequencies `xi` and bandwidths `sigma` for `J*Q` Morlet
    wavelets, `Q` per octave, from `high_freq` down. Adjacent filters cross
    at fraction `r` of their peak (kymatio's rule)."""
    xi = high_freq * 2.0 ** (-torch.arange(J * Q, dtype=torch.float64) / Q)
    factor = 2.0 ** (-1.0 / Q)
    sigma = xi * (1 - factor) / (1 + factor) / math.sqrt(2 * math.log(1 / r))
    return xi, sigma


def morlet_bank(N, J, Q=1, high_freq=0.425, device=None):
    """`[J*Q + 1, N]` complex64 Fourier-domain filters: `J*Q` Morlet
    band-passes (highest frequency first) then the Gaussian low-pass as the
    last row. `N` is the padded length the transform will run at."""
    xi, sigma = morlet_params(J, Q, high_freq)
    psi = torch.stack([morlet_hat(N, float(x), float(s)) for x, s in zip(xi, sigma)])
    phi = gauss_hat(N, float(sigma[-1]))
    return torch.cat([psi, phi[None]]).to(torch.complex64).to(device)

In [ ]:
xi, sigma = morlet_params(J=4, Q=2)
assert len(xi) == 8 and torch.all(xi[1:] < xi[:-1]), "descending frequency"
assert abs(xi[2] / xi[0] - 0.5) < 1e-12, "Q=2: two filters per octave"
bank = morlet_bank(128, J=3, Q=1)
assert bank.shape == (4, 128) and bank.dtype == torch.complex64
# L1-normalized in time => |psi_hat| <= 1 everywhere; and the bank tiles the
# covered band with no gaps (Littlewood-Paley sum stays well above zero)
assert bank.abs().max() <= 1 + 1e-5
f = torch.fft.fftfreq(128)
lp = (bank.abs() ** 2).sum(0)
covered = (f > 0.425 * 2 ** -3) & (f < 0.425)
assert lp[covered].min() > 0.2, lp[covered].min()

## Wavelet transform

Convolution is a product in Fourier. The signal is **reflection-padded to
`2T`** first (Morel's choice) so the last day is not convolved with the first
day -- a periodic boundary would leak the pandemic into last week. The filter
bank is built at the padded length and cached per `(N, J, Q, device)`, since
building it is the only slow part.

Output is complex: modulus carries the envelope at each scale, phase carries
the oscillation. Scattering only keeps the modulus.

In [ ]:
#| export
_BANK_CACHE = {}


def get_bank(N, J, Q=1, high_freq=0.425, device=None):
    """`morlet_bank` with a cache -- the same filters are reused for every
    channel and every scattering order."""
    key = (N, J, Q, high_freq, str(device))
    if key not in _BANK_CACHE:
        _BANK_CACHE[key] = morlet_bank(N, J, Q, high_freq, device=device)
    return _BANK_CACHE[key]


def wavelet_transform(X, bank, rpad=True):
    """`[..., T]` real -> `[..., F, T]` complex: each of the `F` filters in
    `bank` (built at length `2T` if `rpad` else `T`) applied along the last
    axis by FFT."""
    T = X.shape[-1]
    Xp = torch.cat([X, X.flip(-1)], dim=-1) if rpad else X
    assert Xp.shape[-1] == bank.shape[-1], f"bank built for {bank.shape[-1]}, got {Xp.shape[-1]}"
    Xh = torch.fft.fft(Xp.to(torch.float32))
    Y = torch.fft.ifft(Xh.unsqueeze(-2) * bank)
    return Y[..., :T]

In [ ]:
torch.manual_seed(0)
T = 200
bank = get_bank(2 * T, J=4, Q=1)
assert get_bank(2 * T, J=4, Q=1) is bank, "cached"

t = torch.arange(T, dtype=torch.float32)
x = torch.sin(2 * math.pi * 0.1 * t)          # a pure tone at 0.1 cycles/sample
Y = wavelet_transform(x, bank)
assert Y.shape == (5, T) and Y.is_complex()
# the tone lands in the band whose centre frequency is nearest 0.1
xi, _ = morlet_params(4, 1)
expect = int((xi - 0.1).abs().argmin())
energy = (Y[:-1].abs() ** 2).mean(-1)
assert energy.argmax() == expect, (energy, expect)
# batched: leading dims pass through
Yb = wavelet_transform(torch.randn(3, 2, T), bank)
assert Yb.shape == (3, 2, 5, T)

## Scattering

With $U_1 = |x \star \psi_{j_1}|$ and $U_2 = |U_1 \star \psi_{j_2}|$:

$$S_0 = x \star \phi,\qquad S_1[j_1] = U_1[j_1] \star \phi,\qquad
S_2[j_1, j_2] = U_2[j_1, j_2] \star \phi,\quad j_2 > j_1 .$$

Second-order paths only go to **lower** frequencies ($j_2 > j_1$, i.e.
$\xi_{j_2} < \xi_{j_1}$): the modulus $U_1[j_1]$ is an envelope, which lives
below $\xi_{j_1}$, so filtering it at a higher frequency would just be noise.
This halves the path count for free.

`scattering` returns the coefficients **with the time axis intact**
(`[..., n_paths, T]`), not time-averaged -- the low-pass $\phi$ has already
smoothed them, so they can be subsampled, but here the day axis is kept so the
result lines up with the `[channels, days]` arrays from `01_data`. Time-average
yourself (`S.mean(-1)`) when a per-channel descriptor is what you want; see
`scattering_features`.

The path index is returned alongside, as tuples: `()` for $S_0$, `(j1,)` for
first order, `(j1, j2)` for second.

In [ ]:
#| export
def scattering_paths(J, Q=1, order=2):
    """Path tuples in the order `scattering` stacks them."""
    JQ = J * Q
    paths = [()]
    if order >= 1:
        paths += [(j1,) for j1 in range(JQ)]
    if order >= 2:
        paths += [(j1, j2) for j1 in range(JQ) for j2 in range(j1 + 1, JQ)]
    return paths


def scattering(X, J, Q=1, order=2, high_freq=0.425, rpad=True):
    """Wavelet scattering along the last axis. `[..., T]` real ->
    `(S, paths)` with `S` real `[..., n_paths, T]` (time axis kept) and
    `paths` from `scattering_paths`. Stays on `X`'s device."""
    assert order in (0, 1, 2)
    T = X.shape[-1]
    N = 2 * T if rpad else T
    bank = get_bank(N, J, Q, high_freq, device=X.device)
    psi, phi = bank[:-1], bank[-1]
    X = X.to(torch.float32)

    out = [wavelet_transform(X, phi[None], rpad).real]              # S0: [..., 1, T]
    if order >= 1:
        U1 = wavelet_transform(X, psi, rpad).abs()                   # [..., JQ, T]
        out.append(wavelet_transform(U1, phi[None], rpad).real[..., 0, :])
    if order >= 2:
        JQ = psi.shape[0]
        for j1 in range(JQ - 1):
            U2 = wavelet_transform(U1[..., j1, :], psi[j1 + 1:], rpad).abs()   # [..., JQ-j1-1, T]
            out.append(wavelet_transform(U2, phi[None], rpad).real[..., 0, :])
    return torch.cat(out, dim=-2), scattering_paths(J, Q, order)


def scattering_features(X, J, Q=1, order=2, log=True, eps=1e-6, **kw):
    """Time-averaged scattering coefficients: `[..., T]` -> `[..., n_paths]`.
    `log=True` (the usual choice) compresses the dynamic range across scales so
    no single path dominates a distance or correlation."""
    S, paths = scattering(X, J, Q, order, **kw)
    F = S.mean(-1)
    return (torch.log(F.clamp_min(eps)) if log else F), paths

In [ ]:
torch.manual_seed(1)
T, J = 128, 3
x = torch.randn(4, T)
S, paths = scattering(x, J=J, Q=1, order=2)
assert len(paths) == 1 + J + J * (J - 1) // 2 == 7
assert S.shape == (4, 7, T) and not S.is_complex()
assert paths[0] == () and paths[1] == (0,) and paths[-1] == (J - 2, J - 1)
assert torch.all(S[:, 1:] >= 0), "moduli, low-passed, stay non-negative"

# order controls how many paths come back
S1, p1 = scattering(x, J=J, order=1)
assert S1.shape[-2] == 1 + J and torch.allclose(S1, S[:, :1 + J])
S0, _ = scattering(x, J=J, order=0)
assert S0.shape[-2] == 1

# time-averaged features: a shift-invariance sanity check. Two crops of the
# same stationary signal, offset by a few samples, should have descriptors
# that agree far better than the raw samples do.
y = torch.randn(1, 600) * (1 + torch.sin(torch.arange(600.) / 20).abs())
a, b = y[:, :560], y[:, 7:567]
f_a, _ = scattering_features(a, J=4, log=False)
f_b, _ = scattering_features(b, J=4, log=False)
rel_feat = ((f_a - f_b).abs() / (f_a.abs() + 1e-6)).max()
rel_raw = ((a - b).abs().mean() / a.abs().mean())
assert rel_feat < 0.2 and rel_feat < rel_raw / 3, (rel_feat, rel_raw)
fl, _ = scattering_features(a, J=4, log=True)
assert torch.isfinite(fl).all()

## Scattering as a pairing metric

`01_data` builds its channel-similarity matrix from returns directly
(`build_corr`, `build_xcorr`). The time-averaged scattering vector is another
thing to correlate: two channels are "close" if they have the same multi-scale
volatility structure, regardless of whether their day-to-day moves line up.
That is a genuinely different notion of similarity from Pearson-on-returns --
two uncorrelated stocks with the same volatility signature come out adjacent.

`scattering_corr` is a drop-in for `build_corr` (same `[C, C]`, `use_abs`
convention), so every ordering and the benchmark in `01_data` can consume it
unchanged.

In [ ]:
#| export
def scattering_corr(X, J, Q=1, order=2, use_abs=True, **kw):
    """`[C, T]` -> `[C, C]` correlation between channels' log-scattering
    descriptors. Drop-in for `build_corr` / `build_xcorr`."""
    F, _ = scattering_features(X, J, Q, order, log=True, **kw)
    C = torch.nan_to_num(torch.corrcoef(F), nan=0.0)
    return C.abs() if use_abs else C

In [ ]:
torch.manual_seed(2)
T = 256
t = torch.arange(T, dtype=torch.float32)
calm = torch.randn(2, T) * 0.3
bursty = torch.randn(2, T) * (1 + 3 * (torch.sin(2 * math.pi * t / 64) > 0.9).float())
X = torch.cat([calm, bursty])
C = scattering_corr(X, J=4)
assert C.shape == (4, 4) and torch.allclose(C, C.T, atol=1e-5)
assert torch.allclose(C.diag(), torch.ones(4), atol=1e-4)
# same-regime channels are more alike in scattering space than cross-regime
within = (C[0, 1] + C[2, 3]) / 2
across = C[:2, 2:].mean()
assert within > across, (within, across)

## Plotting

`show_scattergram` is the one new view: for a single channel, paths down the
y-axis and days along x. Everything else in `01_data`'s plotting kit already
works on a `[C, T]` slice `S[:, p, :]`, which is the point of keeping the time
axis.

In [ ]:
#| export
def show_scattergram(S, paths, title, log=True, eps=1e-6, cmap="viridis", figsize=(12, 4)):
    """One channel's scattering coefficients as an image: `S` is `[n_paths, T]`."""
    import matplotlib.pyplot as plt
    Z = S.detach().cpu()
    Z = torch.log(Z.clamp_min(eps)) if log else Z
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(Z.numpy(), aspect="auto", cmap=cmap, interpolation="nearest")
    ax.set_yticks(range(len(paths)))
    ax.set_yticklabels([str(p) if p else "S0" for p in paths], fontsize=7)
    ax.set_xlabel("day"); ax.set_ylabel("path (j1, j2)")
    ax.set_title(title + (" (log)" if log else ""))
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.close(fig)
    return fig

In [ ]:
import matplotlib.pyplot as plt
torch.manual_seed(3)
S, paths = scattering(torch.randn(1, 300).cumsum(-1), J=4, Q=1)
fig = show_scattergram(S[0], paths, "random walk")
assert isinstance(fig, plt.Figure) and len(fig.axes) >= 1
fig

## The array, with a third axis

Stacking scattering over the `01_data` orderings gives the full
`[views, channels, paths, days]` array. `stack_views` is a two-liner, but a
named function pins the axis convention down in one place, since the
hypercomplex layers in `03_hypercomplex` depend on it: **views first**, so
they can be read as the components of an `n`-dimensional hypercomplex number.

In [ ]:
#| export
def stack_views(X, orders):
    """`[C, ...]` data + `n` orderings -> `[n, C, ...]`: the same array under
    each ordering, views stacked as the leading axis."""
    return torch.stack([X[o] for o in orders])

In [ ]:
X = torch.arange(12.).reshape(4, 3)
V = stack_views(X, [torch.tensor([0, 1, 2, 3]), torch.tensor([3, 2, 1, 0])])
assert V.shape == (2, 4, 3) and torch.equal(V[1, 0], X[3])
# with scattering: [views, channels, paths, days]
S, paths = scattering(torch.randn(4, 64), J=3)
V = stack_views(S, [torch.arange(4), torch.arange(4).flip(0)])
assert V.shape == (2, 4, len(paths), 64)

## Pipeline

Runs on the real data from `01_data`. Marked `#| eval: false` because it
needs the cached `.pt` files.

In [ ]:
#| eval: false
from shakeqr.data import *
opening, closing, tickers = get_nyse_data()
prices, returns, tk = prepare(opening, closing, tickers)
Zl = log_prices(prices)                     # [C, T], standardized log-prices
J, Q = 6, 1
S, paths = scattering(Zl, J=J, Q=Q)          # [C, n_paths, T]
print(S.shape, len(paths), "paths", S.device)

In [ ]:
#| eval: false
show_scattergram(S[0], paths, f"S&P 500 log-price scattering (J={J}, Q={Q})")

In [ ]:
#| eval: false
# scattering-space pairing vs. return correlation, on the same channel set
returns_w = standardize(returns)
C_ret = build_corr(returns_w)
C_sc = scattering_corr(Zl, J=J, Q=Q)
D_sc = corr_to_dist(C_sc)
order_sc = two_opt(D_sc, greedy_edge_order(D_sc))
order_rt = two_opt(corr_to_dist(C_ret), greedy_edge_order(corr_to_dist(C_ret)))
print(f"roughness on whitened returns -- tsp(returns): {roughness(returns_w, order_rt):.4f}   "
      f"tsp(scattering): {roughness(returns_w, order_sc):.4f}")

In [ ]:
#| eval: false
# the S1 envelope at one scale, all channels, under the scattering-based ordering
p = paths.index((2,))
show(S[order_sc, p, :], f"S1 path {paths[p]} | tsp(scattering)", clip=3.0)

In [ ]:
#| eval: false
# the full array: 4 orderings as views
D_ret = corr_to_dist(C_ret)
orders = {"greedy": greedy_order(C_ret), "spectral": spectral_order(C_ret),
          "tsp": order_rt, "hrp": hrp_order(D_ret)}
V = stack_views(S, list(orders.values()))
print("views x channels x paths x days:", tuple(V.shape))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()